# L'emprise d'évaluation change le classement

Ce carnet rejoue la section 6, qui est le résultat de méthode de
l'article : **une AUC n'a de sens qu'accompagnée de la population sur laquelle
elle a été calculée.**

Deux tables :

- `exp43_fusion_ciblee_coherence.csv` — les AUC par produit sur l'emprise
  commune aux produits à large couverture, 148 159 bâtiments ;
- `exp42_aoi_commun_face_a_face.csv` — le face-à-face, produit par produit, sur
  les seuls bâtiments que les deux couvrent.

In [ ]:
import os
import pandas as pd

# les carnets se lancent depuis notebooks/ ; les tables sont dans data/results
RESULTATS = os.path.join("..", "data", "results")
TABLES = os.path.join("..", "tables")

def lire(dossier, nom, **kw):
    """Lit une table du depot. Le separateur et la virgule decimale varient
    d'une campagne a l'autre : on laisse pandas les deviner sauf indication."""
    kw.setdefault("sep", None)
    kw.setdefault("engine", "python")
    return pd.read_csv(os.path.join(dossier, nom), **kw)

def presque(a, b, tol=0.0006):
    """Egalite au millieme pres, tolerance par defaut d'un demi-millieme."""
    assert abs(a - b) < tol, "%.4f attendu, %.4f obtenu" % (b, a)
    return True


Le classement sur l'emprise strictement commune.

In [ ]:
e = lire(RESULTATS, "exp43_fusion_ciblee_coherence.csv")
commun = e[e.lecture == "A - emprise des huit"].copy()
seuls = commun[commun.score.str.endswith("seul")].sort_values("moyenne",
                                                             ascending=False)
print(seuls[["score", "auc_ems", "auc_chatmap", "moyenne"]]
      .to_string(index=False))
print("\nbatiments de l'emprise commune :", int(commun.n_emprise.iloc[0]))

Les trois chiffres que la page d'accueil met en avant.

In [ ]:
# les chiffres de tete de l'article
tstat = seuls.loc[seuls.score == "notre T-stat seul", "moyenne"].iloc[0]
osu = seuls.loc[seuls.score == "OSU seul", "moyenne"].iloc[0]
presque(tstat, 0.7403)
presque(osu, 0.7324)
assert int(commun.n_emprise.iloc[0]) == 148159
print("Verifie : T-stat 0,7403 ; OSU 0,7324 ; 148 159 batiments.")

Le face-à-face, où les écarts sont les plus larges — et les échantillons les plus inégaux.

In [ ]:
ff = lire(RESULTATS, "exp42_aoi_commun_face_a_face.csv")
ff = ff.sort_values("ecart_chatmap", ascending=False)
print(ff[["adversaire", "n_commun", "auc_tstat_ems", "auc_adversaire_ems",
          "auc_tstat_chatmap", "auc_adversaire_chatmap"]].to_string(index=False))

Pourquoi un face-à-face ne se résume pas en un seul nombre.

In [ ]:
# la comparaison en face-a-face n'est equitable que paire par paire : les
# effectifs varient de deux ordres de grandeur.
print("batiments communs : de %d a %d selon l'adversaire"
      % (ff.n_commun.min(), ff.n_commun.max()))